# Nền tảng 4 — Tokenizer: byte-BPE, SuperBPE, WordPiece, Unigram

Notebook 01 đã dựng BPE đồ chơi để thấy thuật toán gộp cặp. Notebook này đi xa hơn theo ba hướng mà báo cáo cần:

1. **Cấu tạo đầy đủ của một tokenizer thật** — bảng chữ cái byte, pretokenizer regex, danh sách merge — và hai
   cái bẫy trong đó đã suýt làm hỏng số liệu của project.
2. **SuperBPE** — điểm chuyển tiếp, merge kế thừa, superword học được, và nó nén thêm bao nhiêu.
3. **Các họ thuật toán khác** — WordPiece và Unigram/SentencePiece: tiêu chí chọn merge khác nhau ở chỗ nào, và
   vì sao Unigram có thể cho *nhiều* cách tách cho cùng một câu trong khi BPE chỉ có một.

Cuối notebook là phần đo tokenizer **không cần model**: fertility, token/âm tiết, tỷ lệ superword, CTC trên
FLORES-200 — bộ chỉ số dùng để so với bài báo MonTok.

**Chạy bằng kernel pixi của project.** Các cell huấn luyện chạy trên ~1,6MB văn bản, mất vài giây.

In [ ]:
import json
import math
import os
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import regex as re
from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "kaggle" / "outputs" / "vitok-data"
WORK = Path(os.environ.get("TMPDIR", "/tmp")) / "vitok_nb04"
WORK.mkdir(exist_ok=True)

from vitok import superbpe
from vitok.tokenizer_spec import SPECIAL_TOKENS, STAGE1_REGEX, STAGE2_REGEX
from vitok.train_tokenizers import _pre_tokenizer

docs = [json.loads(l)["text"] for l in (DATA / "test.jsonl").open(encoding="utf-8")]
mini = "\n".join(docs[:600])
mini_path = WORK / "mini_nfc.txt"
mini_path.write_text(unicodedata.normalize("NFC", mini), encoding="utf-8")
print(f"corpus nhỏ để huấn luyện trong notebook: {len(mini):,} ký tự = {len(mini.encode()) / 1e6:.2f} MB")
print(f"(tokenizer thật của project huấn luyện trên 500 MB — xem train_meta.json)")

## 1. Tokenizer giải quyết bài toán gì

Model chỉ nhận dãy số nguyên. Có ba cách biến văn bản thành số, và mỗi cách hỏng ở một đầu:

| Đơn vị | Kích thước vocab | Độ dài chuỗi | Vấn đề |
|---|---|---|---|
| Ký tự / byte | 256 | dài nhất | mỗi bước học được rất ít; ngữ cảnh $T$ token chứa rất ít nội dung |
| Từ (word) | vô hạn trên thực tế | ngắn nhất | luôn gặp từ chưa thấy (OOV); tiếng Việt còn phải định nghĩa "từ" |
| Subword | chọn được, 16k–256k | ở giữa | phải học từ dữ liệu; đây là lựa chọn phổ biến |

Chi phí gắn với lựa chọn này rất cụ thể, và notebook 03 đã cho công thức. Với model d8 và ngữ cảnh $T = 1024$:

- FLOPs mỗi token $\approx 2{,}5 \times 10^{8}$;
- một tokenizer nén tốt hơn 10% thì cùng lượng văn bản tốn ít hơn 10% token, tức ít hơn 10% FLOPs;
- nhưng vocab to hơn làm `lm_head` to ra, nên FLOPs mỗi token lại tăng.

Hai hiệu ứng ngược chiều. Cell dưới tính điểm cân bằng cho d8: tăng vocab từ 16k lên 32k có lợi không, nếu nhờ
đó chars/token tăng như số đo thật của project?

In [ ]:
c16 = json.loads((DATA / "compression-16k.json").read_text())
c32 = json.loads((DATA / "compression-32k.json").read_text())

d, L, T, h, dh = 512, 8, 1024, 4, 128


def flops_moi_ky_tu(vocab_size, chars_per_token):
    v = ((vocab_size + 9 + 63) // 64) * 64
    n_matmul = 12 * d * d * L + d * v                    # thân + lm_head
    f_token = 6 * n_matmul + 12 * h * dh * T * L
    return f_token / chars_per_token                      # quy về MỖI KÝ TỰ để so công bằng


print(" tokenizer      | vocab | chars/token | FLOPs/token   | FLOPs/ký tự")
for ten, c, v in (("bpe-nfc 16k", c16, 16000), ("bpe-nfc 32k", c32, 32000),
                  ("super-nfc 16k", c16, 16000), ("super-nfc 32k", c32, 32000)):
    cond = "bpe-nfc" if ten.startswith("bpe") else "super-nfc"
    cpt = c[cond]["chars_per_token"]
    vv = ((v + 9 + 63) // 64) * 64
    f_tok = 6 * (12 * d * d * L + d * vv) + 12 * h * dh * T * L
    print(f" {ten:14s} | {v // 1000:3d}k  | {cpt:11.3f} | {f_tok:13,.0f} | {f_tok / cpt:11,.0f}")

print("\nCột cuối là cột đáng nhìn: nó trả lời 'xử lý MỘT KÝ TỰ văn bản tốn bao nhiêu phép tính'.")

## 2. Bảng chữ cái byte và ánh xạ của GPT-2

Nếu bảng chữ cái gốc là **ký tự Unicode** thì vocab phải chứa mọi ký tự có thể gặp — hàng trăm nghìn — và vẫn
gãy khi gặp ký tự lạ. Byte-level BPE giải quyết triệt để: bảng chữ cái gốc là **256 byte**, nên mọi văn bản đều
mã hoá được, không bao giờ có OOV.

Rắc rối kỹ thuật: nhiều byte không phải ký tự in được, nên `tokenizers` dùng ánh xạ của GPT-2 để đưa 256 byte về
256 ký tự Unicode in được. Byte 32 (dấu cách) thành `Ġ`, byte 10 (xuống dòng) thành `Ċ`. Đó là lý do khi in vocab
bạn thấy `Ġcủa` chứ không phải ` của`.

Hệ quả cho tiếng Việt: một ký tự tiếng Việt có dấu chiếm 2–3 byte UTF-8, nên ở mức byte thì "ố" là ba token
riêng trước khi được gộp lại.

In [ ]:
def bang_byte_unicode():
    """Ánh xạ byte -> ký tự in được của GPT-2: giữ nguyên các byte đã in được, đẩy phần còn lại lên U+0100."""
    bs = (list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1))
          + list(range(ord("®"), ord("ÿ") + 1)))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, (chr(c) for c in cs)))


B2U = bang_byte_unicode()
U2B = {v: k for k, v in B2U.items()}


def doc_token(t):
    """Đưa một token byte-level về dạng người đọc được."""
    return bytes(U2B[c] for c in t if c in U2B).decode("utf-8", errors="replace")


print(f"kích thước bảng chữ cái byte-level: {len(pre_tokenizers.ByteLevel.alphabet())}")
print("vài ánh xạ byte -> ký tự hiển thị:",
      {f"byte {b} ({name})": B2U[b] for b, name in ((32, "dấu cách"), (10, "xuống dòng"), (65, "A"), (97, "a"))})

for s in ("của", "ố", "Việt"):
    b = s.encode("utf-8")
    print(f"  {s!r}: {len(s)} ký tự NFC, {len(b)} byte UTF-8 -> {list(b)}")

pre = _pre_tokenizer(STAGE1_REGEX)
print("\npretokenizer đầy đủ áp lên một câu (trái: byte-level, phải: đọc được):")
for t, _ in pre.pre_tokenize_str("Chính phủ đã ban hành."):
    print(f"    {t:<14s} | {doc_token(t)!r}")

### Bẫy 1: bảng chữ cái không đủ 256

`BpeTrainer` mặc định chỉ đưa vào bảng chữ cái những ký tự **xuất hiện trong corpus huấn luyện**. Byte nào không
gặp sẽ không có id, và lúc encode văn bản mới chứa byte đó, token bị **bỏ im lặng**. Hệ quả với chỉ số của
project: văn bản test mất vài byte khó → tổng nat nhỏ đi → bpc trông đẹp hơn thực tế, mà không có thông báo lỗi
nào.

Vì vậy `vitok/train_tokenizers.py` truyền `initial_alphabet=pre_tokenizers.ByteLevel.alphabet()`:

```python
trainer = BpeTrainer(vocab_size=vocab_size, show_progress=True,
                     initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
```

Cell dưới dựng lại đúng lỗi đó trên corpus chỉ có tiếng Việt, rồi encode một câu có ký tự Nhật.

In [ ]:
def huan_luyen_bpe(files, vocab_size, regex=STAGE1_REGEX, day_du_256=True):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = _pre_tokenizer(regex)
    kw = {"initial_alphabet": pre_tokenizers.ByteLevel.alphabet()} if day_du_256 else {}
    tok.train([str(f) for f in files], trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False, **kw))
    tok.decoder = decoders.ByteLevel()
    return tok


# corpus 20.000 ký tự tiếng Việt: nhiều byte UTF-8 không hề xuất hiện
tiny_path = WORK / "tiny_nfc.txt"
tiny_path.write_text(unicodedata.normalize("NFC", "\n".join(docs[:5]))[:20000], encoding="utf-8")
bang_256 = set(pre_tokenizers.ByteLevel.alphabet())

la = "Giá 1000円 tại 東京 — hết"
for ten, day_du in (("thiếu bảng chữ cái", False), ("đủ 256 byte", True)):
    t = huan_luyen_bpe([tiny_path], 1000, day_du_256=day_du)
    co = len(bang_256 & set(t.get_vocab()))
    e = t.encode(la, add_special_tokens=False)
    lai = t.decode(e.ids)
    print(f"{ten:20s}: {co:3d}/256 byte có id | {len(e.ids):2d} token")
    print(f"{'':20s}  giải mã lại: {lai!r}  {'<- MẤT CHỮ, im lặng' if lai != la else '<- khớp'}")

## 3. Pretokenizer: quyết định cái gì **không bao giờ** được gộp

Trước khi BPE chạy, văn bản bị cắt thành các **pretoken** bằng một biểu thức chính quy. BPE chỉ gộp *bên trong*
một pretoken, nên regex này đặt ra giới hạn cứng cho mọi token về sau.

Regex stage 1 của project (`vitok/tokenizer_spec.py`) gồm các nhánh:

| Nhánh | Bắt cái gì |
|---|---|
| `[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+` | (dấu cách tuỳ chọn) + chữ, kiểu " của" |
| `\p{N}{1,3}` | tối đa 3 chữ số một cụm |
| ` ?[^\s\p{L}\p{N}]+[\r\n/]*` | cụm dấu câu |
| `\s*[\r\n]+`, `\s+(?!\S)`, `\s+` | khoảng trắng và xuống dòng |

Điểm quyết định: nhánh chữ bắt đầu bằng `[^\r\n\p{L}\p{N}]?`, tức **một dấu cách được phép đi kèm vào đầu từ**.
Nhờ vậy " của" là một pretoken, còn "của" (không có dấu cách) là pretoken khác — hai token khác nhau trong vocab.
Nhưng regex **không cho phép** một pretoken chứa dấu cách ở *giữa*, nên với regex này không token nào bắc qua
hai từ. Đó chính là ràng buộc mà SuperBPE gỡ bỏ ở mục 5.

### Bẫy 2: thiếu `\p{M}` thì NFD vỡ

`\p{M}` là lớp **dấu phụ kết hợp** (combining mark). Ở dạng NFD, "ố" là ba code point: `o` + dấu mũ + dấu sắc.
Nếu lớp chữ trong regex chỉ có `\p{L}` (như pattern gốc của nanochat), hai dấu phụ **không** thuộc lớp chữ, nên
chúng rơi vào nhánh dấu câu và bị tách khỏi chữ cái nền.

Hậu quả: token của điều kiện NFD sẽ là "o" + "dấu rời", tokenizer trông vẫn chạy nhưng toàn bộ so sánh NFC/NFD
mất ý nghĩa. Cell dưới đặt cạnh nhau hai pattern trên cùng một chuỗi NFD.

In [ ]:
NANOCHAT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

cau = "thống nhất"
nfd = unicodedata.normalize("NFD", cau)
print(f"NFC: {cau!r} -> {len(cau)} code point")
print(f"NFD: {nfd!r} -> {len(nfd)} code point")
print("các code point NFD:", [unicodedata.name(c, repr(c)).replace("COMBINING ", "◌") for c in nfd][:8], "...\n")

for ten, pat in (("nanochat (thiếu \\p{M})", NANOCHAT_PATTERN), ("vitok stage 1 (có \\p{M})", STAGE1_REGEX)):
    mieng = re.findall(pat, nfd)
    print(f"{ten}:")
    print("   ", [m for m in mieng])
    print(f"    số mảnh: {len(mieng)}")

## 4. BPE trong tokenizer thật của project

Thuật toán vẫn là thuật toán ở notebook 01: đếm mọi cặp token liền nhau, gộp cặp **tần suất cao nhất**, lặp lại.
Cái mới ở bản thật là dữ liệu được nén thành `(pretoken, số lần xuất hiện)` nên mỗi lượt gộp không phải quét lại
toàn corpus.

File `tokenizer.json` lưu hai thứ: `vocab` (token → id) và `merges` (danh sách cặp, **theo đúng thứ tự học**).
Thứ tự này là thứ quyết định lúc encode: áp merge theo đúng thứ tự đã học, không phải theo tham lam tại chỗ.

In [ ]:
bpe16 = Tokenizer.from_file(str(DATA / "tokenizers-16k" / "bpe-nfc" / "tokenizer.json"))
sup16 = Tokenizer.from_file(str(DATA / "tokenizers-16k" / "super-nfc" / "tokenizer.json"))
meta = json.loads((DATA / "tokenizers-16k" / "train_meta.json").read_text())

raw = json.loads(Path(DATA / "tokenizers-16k" / "bpe-nfc" / "tokenizer.json").read_text())
merges_bpe = raw["model"]["merges"]
print(f"vocab bpe-nfc 16k : {bpe16.get_vocab_size()} (gồm {len(SPECIAL_TOKENS)} special token)")
print(f"số merge          : {len(merges_bpe)}")
print(f"bảng chữ cái gốc  : {bpe16.get_vocab_size() - len(merges_bpe) - len(SPECIAL_TOKENS)} token\n")

def doc_merge(m):
    return m if isinstance(m, str) else " ".join(m)

print("20 merge đầu tiên (học sớm nhất = cặp phổ biến nhất).")
print("Cột trái là chuỗi byte-level lưu trong file, cột phải là dạng đọc được:")
for i, m in enumerate(merges_bpe[:20]):
    a, b = doc_merge(m).split(" ")
    print(f"  {i:3d}: {a:>8s} + {b:<8s} -> {(a + b):<12s} | {doc_token(a + b)!r}")

print("\n5 merge cuối cùng (học muộn nhất = hiếm nhất trong các cặp đáng gộp):")
for i, m in enumerate(merges_bpe[-5:], start=len(merges_bpe) - 5):
    a, b = doc_merge(m).split(" ")
    print(f"  {i:3d}: {a:>8s} + {b:<8s} -> {(a + b):<12s} | {doc_token(a + b)!r}")

print("\nMerge số 0-1 là hai nửa của một ký tự tiếng Việt có dấu (3 byte UTF-8), chưa phải chữ;")
print("phải vài chục merge nữa mới ra được âm tiết hoàn chỉnh. Đó là cái giá của byte-level cho tiếng Việt.")

## 5. SuperBPE: bỏ ràng buộc "không bắc qua dấu cách"

Ý tưởng của SuperBPE (arXiv 2503.13423) chỉ gồm một thay đổi: **huấn luyện hai giai đoạn**.

- **Giai đoạn 1** dùng regex có tách theo dấu cách, học $t \cdot V$ merge đầu tiên. Với $t = 0{,}9$ và
  $V = 16\,000$, đó là $14\,400 - 256 = 14\,144$ merge (256 là bảng chữ cái byte).
- **Giai đoạn 2** đổi sang regex **không** tách theo dấu cách đơn, **kế thừa nguyên vẹn** các merge của giai đoạn
  1, rồi học tiếp $0{,}1 \cdot V = 1\,600$ merge nữa. Các merge mới này được phép bắc qua dấu cách, tạo ra
  **superword** như " có thể", " sử dụng".

Vì sao hai giai đoạn thay vì bỏ tách dấu cách ngay từ đầu: bài báo SuperBPE mô tả đây là một **lịch học**
(curriculum). Nếu bỏ tách dấu cách từ bước đầu, các cụm nhiều từ rất phổ biến sẽ cạnh tranh ngân sách merge với
chính các mảnh từ, nên tokenizer có thể học superword trước khi học xong từ vựng cấp từ. Giai đoạn 1 buộc nó
hoàn thành phần từ vựng cấp từ trước; tham số $t$ quyết định chia ngân sách thế nào, và bài báo khảo sát nhiều
giá trị $t$ thay vì coi 0,9 là tối ưu tuyệt đối.

Chi tiết thứ hai, nằm trong `_breaks_rules` của `vitok/superbpe.py` (chép đúng luật của fork SuperBPE): một token
bị **loại** nếu nó chứa quá **4 từ**, hoặc chứa chuỗi `:Ġ` (dấu hai chấm theo sau là dấu cách). Luật thứ nhất
chặn các cụm dài; luật thứ hai chặn các token kiểu "Ghi chú: " dính nhãn với nội dung phía sau.

Cell dưới huấn luyện thật một cặp BPE/SuperBPE nhỏ (vocab 3.000, $t = 0{,}9$) trên 1,6MB để bạn nhìn thấy toàn bộ
quy trình chạy trong vài giây.

In [ ]:
VOCAB_NHO, T_CHUYEN = 3000, 0.9

t0 = time.time()
bpe_nho = huan_luyen_bpe([mini_path], VOCAB_NHO)
cwd = os.getcwd()
os.chdir(WORK)
bpe_nho.model.save(".")                      # xuất merges.txt + vocab.json để giai đoạn 2 kế thừa
os.chdir(cwd)
print(f"giai đoạn 1 xong trong {time.time() - t0:.1f}s")

merges_txt = (WORK / "merges.txt").read_text(encoding="utf-8").splitlines()[1:]
vocab_txt = json.loads((WORK / "vocab.json").read_text(encoding="utf-8"))
n_alpha = len(vocab_txt) - len(merges_txt)
n_ke_thua = round(T_CHUYEN * VOCAB_NHO) - n_alpha
print(f"bảng chữ cái: {n_alpha} | merge giai đoạn 1: {len(merges_txt)} | kế thừa sang giai đoạn 2: {n_ke_thua}")

ke_thua = [tuple(m.split(" ")) for m in merges_txt[:n_ke_thua]]
vocab_ke_thua = {t: i for t, i in vocab_txt.items() if i < n_alpha + len(ke_thua)}

t0 = time.time()
ids = superbpe.encode_corpus([str(mini_path)], vocab_ke_thua, ke_thua, _pre_tokenizer(STAGE2_REGEX))
t_encode = time.time() - t0
t0 = time.time()
vocab_day_du, merge_moi = superbpe.train_stage2(ids, vocab_ke_thua, VOCAB_NHO, log_every=10 ** 9)
print(f"giai đoạn 2: encode {len(ids):,} id trong {t_encode:.1f}s, học {len(merge_moi)} merge mới trong {time.time() - t0:.1f}s")

sup_nho = Tokenizer(models.BPE(vocab=vocab_day_du, merges=ke_thua + merge_moi))
sup_nho.pre_tokenizer = _pre_tokenizer(STAGE2_REGEX)
sup_nho.decoder = decoders.ByteLevel()

Xem thử các superword nó học được. Đây là chỗ thấy rõ nhất SuperBPE làm gì: các merge mới **không** phải mảnh từ,
mà là cụm từ hay đi cùng nhau trong tiếng Việt.

In [ ]:
print(f"{len(merge_moi)} merge mới của giai đoạn 2, 25 cái đầu:")
for i, (a, b) in enumerate(merge_moi[:25]):
    hop = sup_nho.decode([vocab_day_du[a + b]])
    print(f"  {i:3d}: {hop!r}")

co_khoang_trang = [t for t in vocab_day_du if "Ġ" in t.strip("Ġ")]
print(f"\ntoken chứa dấu cách ở giữa (superword): {len(co_khoang_trang)}/{len(vocab_day_du)}")

So sánh trực tiếp hai tokenizer nhỏ vừa huấn luyện trên cùng một đoạn văn bản chưa từng thấy, rồi so với hai
tokenizer thật 16k của project.

In [ ]:
thu = "\n".join(docs[600:700])
mau = "Chính phủ đã ban hành nghị quyết về phát triển kinh tế xã hội trong năm nay."

print("cùng một câu, bốn tokenizer:")
for ten, t in (("bpe 3k (notebook)", bpe_nho), ("super 3k (notebook)", sup_nho),
               ("bpe-nfc 16k (thật)", bpe16), ("super-nfc 16k (thật)", sup16)):
    e = t.encode(mau, add_special_tokens=False)
    manh = [t.decode([i]) for i in e.ids]
    print(f"\n  {ten}: {len(e.ids)} token")
    print("   ", " | ".join(manh))

print("\n\nnén trên 100 văn bản test chưa dùng để huấn luyện:")
print("  tokenizer            | token   | chars/token | so với BPE cùng cỡ")
goc = {}
for ten, t, nhom in (("bpe 3k", bpe_nho, "3k"), ("super 3k", sup_nho, "3k"),
                     ("bpe-nfc 16k", bpe16, "16k"), ("super-nfc 16k", sup16, "16k")):
    n_tok = sum(len(t.encode(d, add_special_tokens=False).ids) for d in docs[600:700])
    cpt = len(thu) / n_tok
    goc.setdefault(nhom, cpt)
    print(f"  {ten:20s} | {n_tok:7,} | {cpt:11.3f} | {cpt / goc[nhom] - 1:+18.1%}")

## 6. WordPiece: đổi tiêu chí chọn cặp

BPE chọn cặp có **tần suất** cao nhất. WordPiece (Schuster & Nakajima 2012, dùng trong BERT) chọn cặp làm tăng
**likelihood** của một mô hình ngôn ngữ trên corpus nhiều nhất. Bài gốc không công bố đủ chi tiết; mô tả phổ
biến nhất (giáo trình NLP của Hugging Face) rút tiêu chí đó về điểm số dưới đây. Lưu ý thực tế:
`WordPieceTrainer` trong thư viện `tokenizers` lại huấn luyện bằng chính thuật toán BPE và chỉ khác ở định dạng
vocab (tiền tố `##`) và cách encode, nên điểm số này là mô tả thuật toán, không phải thứ thư viện đó chạy

$$\mathrm{score}(a, b) = \frac{\mathrm{freq}(ab)}{\mathrm{freq}(a)\cdot\mathrm{freq}(b)}$$

**Ký hiệu mới:** $\mathrm{freq}(ab)$ — số lần $a$ đứng ngay trước $b$; $\mathrm{freq}(a)$ — số lần $a$ xuất hiện ở
bất kỳ đâu.

Ý nghĩa: mẫu số phạt các mảnh vốn đã rất phổ biến. Nếu "e" và "s" đều xuất hiện khắp nơi thì cặp "es" dù đếm
được nhiều vẫn có điểm thấp, vì việc chúng đứng cạnh nhau không mang nhiều thông tin — đúng tinh thần
**pointwise mutual information**.

Hệ quả thực tế: WordPiece thiên về các mảnh **đặc trưng** hơn là các mảnh phổ biến. Cell dưới cài cả hai tiêu chí
trên một corpus đồ chơi để thấy chúng chọn hai cặp khác nhau ở cùng một bước.

In [ ]:
corpus_do_choi = {
    "của nhà": 90, "của một": 80, "của tôi": 70, "của bạn": 60,     # "của" đi với đủ thứ
    "một số": 50, "một cách": 40,                                    # "một" cũng vậy
    "kinh tế": 50,                                                   # "kinh"/"tế" gần như chỉ đi với nhau
    "kinh": 10, "tế": 10,                                            # (vài lần đứng một mình)
}

dem_manh, dem_cap = Counter(), Counter()
for cum, so in corpus_do_choi.items():
    manh = cum.split()
    for m in manh:
        dem_manh[m] += so
    for a, b in zip(manh, manh[1:]):
        dem_cap[(a, b)] += so

print(" cặp                  | freq(ab) | freq(a) | freq(b) | điểm WordPiece")
for (a, b), f in dem_cap.most_common(6):
    diem = f / (dem_manh[a] * dem_manh[b])
    print(f" {a + ' + ' + b:20s} | {f:8d} | {dem_manh[a]:7d} | {dem_manh[b]:7d} | {diem:.6f}")

bpe_chon = max(dem_cap.items(), key=lambda kv: kv[1])[0]
wp_chon = max(dem_cap.items(), key=lambda kv: kv[1] / (dem_manh[kv[0][0]] * dem_manh[kv[0][1]]))[0]
print(f"\nBPE gộp      : {bpe_chon}  (tần suất cao nhất)")
print(f"WordPiece gộp: {wp_chon}  (điểm cao nhất)")
print("\n'của một' xuất hiện nhiều gấp đôi 'kinh tế', nhưng 'của' và 'một' đứng cạnh rất nhiều thứ khác,")
print("nên việc chúng cạnh nhau ít mang thông tin. WordPiece phạt đúng chỗ đó qua mẫu số.")

## 7. Unigram / SentencePiece: mô hình xác suất thay vì danh sách merge

Unigram (Kudo 2018, cài trong SentencePiece) đảo ngược hoàn toàn cách nghĩ.

**Mô hình**: vocab là một tập mảnh, mỗi mảnh $x$ có xác suất $p(x)$. Một cách tách $\mathbf{s} = (x_1,\dots,x_k)$
của câu có xác suất

$$P(\mathbf{s}) = \prod_{i=1}^{k} p(x_i), \qquad \text{log} P(\mathbf{s}) = \sum_{i=1}^{k} \log p(x_i)$$

**Vì sao là tích:** Unigram giả định các mảnh độc lập với nhau.

**Encode** = tìm cách tách có $\log P$ lớn nhất. Vì các mảnh nối tiếp nhau, đây là bài toán đường đi dài nhất trên
đồ thị không chu trình, giải bằng **quy hoạch động Viterbi** trong $O(n \cdot \ell_{\max})$: đặt $V[j]$ là
log-xác suất tốt nhất của $j$ ký tự đầu, thì

$$V[j] = \max_{\substack{i < j \\ s[i{:}j] \in \text{vocab}}} \big( V[i] + \log p(s[i{:}j]) \big)$$

**Ký hiệu mới**
- $s[i{:}j]$ — đoạn con từ ký tự $i$ tới trước ký tự $j$ (như cắt lát trong Python)
- $\max$ lấy qua mọi điểm cắt $i < j$ mà đoạn cuối $s[i{:}j]$ có trong vocab
- $n$ — độ dài chuỗi; $\ell_{\max}$ — độ dài mảnh dài nhất trong vocab

> **Ký hiệu $O(\cdot)$ (big-O):** chi phí tăng theo cỡ đầu vào ra sao, bỏ qua hằng số. $O(n)$: gấp đôi $n$ thì gấp
> đôi chi phí. $O(n^2)$: gấp đôi $n$ thì gấp bốn.

**Huấn luyện** đi ngược chiều BPE: bắt đầu từ một vocab **thừa** (mọi chuỗi con hay gặp), rồi lặp EM và **loại
bỏ** một tỷ lệ cố định các mảnh có đóng góp likelihood thấp nhất ở mỗi vòng (SentencePiece mặc định giữ lại 75%,
`shrinking_factor = 0.75`), cho tới khi còn đúng $V$ mảnh. Các mảnh một ký tự luôn được giữ để mọi chuỗi đều
tách được.

Ba khác biệt đáng nhớ so với BPE:

1. BPE cho **một** cách tách; Unigram cho một phân phối trên nhiều cách tách, nên có thể **lấy mẫu** — đó là
   *subword regularization*, một dạng tăng cường dữ liệu.
2. Vocab Unigram không cần đóng theo phép ghép; bỏ một mảnh không phá các mảnh khác.
3. Encode của Unigram là tối ưu toàn cục theo model; encode của BPE là áp merge theo thứ tự đã học.

Cell dưới cài Viterbi từ đầu trên một vocab đồ chơi, và in **mọi** cách tách kèm log-xác suất để thấy Viterbi
chọn đúng cái tốt nhất.

In [ ]:
p_manh = {"chính": 0.030, "phủ": 0.025, "chính phủ": 0.020, "ch": 0.004, "ính": 0.006,
          "p": 0.003, "hủ": 0.005, " ": 0.080}
log_p = {k: math.log(v) for k, v in p_manh.items()}
cau = "chính phủ"


def viterbi(s, log_p):
    n = len(s)
    V = [-math.inf] * (n + 1)
    lui = [None] * (n + 1)
    V[0] = 0.0
    for j in range(1, n + 1):
        for i in range(j):
            manh = s[i:j]
            if manh in log_p and V[i] > -math.inf and V[i] + log_p[manh] > V[j]:
                V[j], lui[j] = V[i] + log_p[manh], i
    if V[n] == -math.inf:
        return None, -math.inf
    ra, j = [], n
    while j > 0:
        ra.append(s[lui[j]:j])
        j = lui[j]
    return ra[::-1], V[n]


def moi_cach_tach(s, log_p):
    if not s:
        yield [], 0.0
        return
    for j in range(1, len(s) + 1):
        if s[:j] in log_p:
            for duoi, lg in moi_cach_tach(s[j:], log_p):
                yield [s[:j]] + duoi, log_p[s[:j]] + lg


print("mọi cách tách hợp lệ của", repr(cau))
for tach, lg in sorted(moi_cach_tach(cau, log_p), key=lambda kv: -kv[1]):
    print(f"  logP = {lg:8.3f}  P = {math.exp(lg):.3e}   {tach}")

tot, lg = viterbi(cau, log_p)
print(f"\nViterbi chọn: {tot}  (logP = {lg:.3f})")

Huấn luyện Unigram thật trên cùng 1,6MB corpus và so với BPE. Lưu ý: SentencePiece truyền thống làm việc trên
**ký tự** với dấu cách đổi thành `▁` (Metaspace), không phải trên byte, nên hai bên khác nhau cả ở tầng biểu diễn
chứ không chỉ ở thuật toán.

In [ ]:
uni = Tokenizer(models.Unigram())
uni.pre_tokenizer = pre_tokenizers.Metaspace()
uni.decoder = decoders.Metaspace()
t0 = time.time()
uni.train([str(mini_path)], trainers.UnigramTrainer(vocab_size=VOCAB_NHO, show_progress=False,
                                                    special_tokens=["<unk>"], unk_token="<unk>"))
print(f"Unigram huấn luyện trong {time.time() - t0:.1f}s, vocab {uni.get_vocab_size()}")

print("\ncùng câu mẫu:")
for ten, t in (("BPE 3k    ", bpe_nho), ("Unigram 3k", uni)):
    e = t.encode(mau, add_special_tokens=False)
    print(f"  {ten}: {len(e.ids):2d} token | {' | '.join(t.decode([i]) for i in e.ids)}")

print("\nnén trên 100 văn bản test:")
for ten, t in (("BPE 3k    ", bpe_nho), ("Unigram 3k", uni)):
    n_tok = sum(len(t.encode(d, add_special_tokens=False).ids) for d in docs[600:700])
    print(f"  {ten}: {n_tok:,} token | {len(thu) / n_tok:.3f} chars/token")
print("\n(cả hai đều huấn luyện trên 1,6MB nên con số tuyệt đối thấp hơn tokenizer 500MB của project;")
print(" điều đáng nhìn là chúng ở cùng một vùng, khác biệt lớn nằm ở thiết kế chứ không ở thuật toán.)")

## 8. Đo một tokenizer khi chưa có model

Bốn chỉ số project dùng, và chúng trả lời câu hỏi khác nhau:

| Chỉ số | Công thức | Trả lời |
|---|---|---|
| **Fertility / chars per token** | $C/T$ | một token chứa bao nhiêu ký tự văn bản |
| **Token mỗi âm tiết** | $T / (\text{số âm tiết})$ | so với đơn vị tự nhiên của tiếng Việt |
| **Tỷ lệ superword** | số lần dùng token chứa dấu cách $/\,T$ | SuperBPE thực sự được dùng bao nhiêu |
| **CTC** (corpus token count) | tổng token trên một corpus chuẩn | so được với con số công bố của bài khác |

CTC là chỉ số để **so với bài báo khác**, vì nó cố định corpus (FLORES-200, split dev+devtest, 2.009 dòng tiếng
Việt) nên hai bên đo trên cùng văn bản. `vitok/flores_ctc.py` tải FLORES và các tokenizer MonTok
(arXiv 2510.21909) rồi đo lại tất cả trong cùng một lần chạy, thay vì tin vào con số in trong bảng.

In [ ]:
ctc = json.loads((ROOT / "results" / "flores_ctc_32k.json").read_text())
print(f"FLORES-200 vie_Latn, {ctc['lines']} dòng, {ctc['chars_nfc']:,} ký tự NFC")
print(f"hiệu chuẩn: đếm lại tokenizer công bố được {ctc['calibration']['counts']['dev+devtest']:,}"
      f" so với {ctc['calibration']['published']:,} in trong bài -> lệch {ctc['calibration']['gap_frac']:.2%}\n")

print(" tokenizer                     | vocab | CTC     | chars/token | giảm so với BPE 16k")
moc = ctc["montok"]["bpe_16384"]["ctc"]
for ten, v in ctc["montok"].items():
    print(f" MonTok {ten:22s} | {v['vocab_size']:5d} | {v['ctc']:7,} | {v['chars_per_token']:11.3f} |"
          f" {1 - v['ctc'] / moc:18.1%}")
for ten, v in ctc["ours"].items():
    if isinstance(v, dict) and "ctc" in v:
        print(f" vitok  {ten:22s} | {v.get('vocab_size', 0):5d} | {v['ctc']:7,} | {v['chars_per_token']:11.3f} |"
              f" {1 - v['ctc'] / moc:18.1%}")

In [ ]:
print("chỉ số nén đo trên shard val 5.000 văn bản (compression-16k.json / -32k.json):\n")
print(" vocab | điều kiện  | chars/token | token/âm tiết | vocab superword | tỷ lệ token superword")
for ten, c in (("16k", c16), ("32k", c32)):
    for cond, v in c.items():
        if not isinstance(v, dict):
            continue
        print(f" {ten:5s} | {cond:10s} | {v['chars_per_token']:11.3f} | {v['tokens_per_syllable']:13.3f} |"
              f" {v['superword_vocab']:15,} | {v['superword_token_share']:21.1%}")

print("\nsuperword hay dùng nhất của super-nfc 16k:")
for t, n in c16["super-nfc"]["top_superwords"][:12]:
    print(f"  {t!r:22s} {n:,}")

`tokens_per_syllable` dưới 1 là dấu hiệu SuperBPE đang làm đúng việc của nó: trung bình **một token phủ nhiều hơn
một âm tiết**. Nhưng chỉ số này chưa nói superword có trùng với **từ thật** trong tiếng Việt hay không — đó là
giả thuyết H4, đo bằng `vitok/wordhood.py` và sẽ nói kỹ ở notebook 05.

In [ ]:
wh = json.loads((ROOT / "results" / "wordhood_16k.json").read_text())
for cond in ("super-nfc", "super-nfd"):
    v = wh[cond]
    print(f"{cond}: {v['superword_occurrences']:,} lần dùng superword | "
          f"trùng ranh giới từ thật {v['match_rate']:.1%} | mốc khớp tần suất {v['frequency_matched_baseline']:.1%} "
          f"| mốc mọi cụm âm tiết {v['baseline_rate']:.1%}")
print("(vì sao phải so với mốc khớp tần suất chứ không phải mốc mọi cụm: notebook 05 mục 8)")

## 9. Không dùng tokenizer thì sao?

Hướng **tokenizer-free** (ByT5, MambaByte, Byte Latent Transformer) bỏ hẳn bước này, cho model đọc thẳng byte.
Ưu điểm: không bao giờ OOV, không có thiên lệch do vocab, xử lý lỗi chính tả tốt hơn. Nhược điểm nằm ở chi phí, và
với tiếng Việt nhược điểm đó nặng hơn tiếng Anh vì ký tự có dấu chiếm 2–3 byte UTF-8.

Cell dưới tính đúng phần chi phí đó trên dữ liệu thật của project.

In [ ]:
mau_van_ban = "".join(docs[600:700])
n_char_nfc = len(unicodedata.normalize("NFC", mau_van_ban))
n_byte_nfc = len(unicodedata.normalize("NFC", mau_van_ban).encode())
n_byte_nfd = len(unicodedata.normalize("NFD", mau_van_ban).encode())

print(f"ký tự NFC      : {n_char_nfc:,}")
print(f"byte UTF-8 NFC : {n_byte_nfc:,}  ({n_byte_nfc / n_char_nfc:.2f} byte mỗi ký tự)")
print(f"byte UTF-8 NFD : {n_byte_nfd:,}  ({n_byte_nfd / n_byte_nfc - 1:+.1%} so với NFC)\n")

cpt_bpe = c16["bpe-nfc"]["chars_per_token"]
cpt_sup = c16["super-nfc"]["chars_per_token"]
print("độ dài chuỗi cho CÙNG một đoạn văn bản 1.000 ký tự:")
print(f"  byte-level (không tokenizer): {1000 * n_byte_nfc / n_char_nfc:7,.0f} bước")
print(f"  BPE 16k                     : {1000 / cpt_bpe:7,.0f} bước  ({n_byte_nfc / n_char_nfc * cpt_bpe:.1f}× ngắn hơn byte)")
print(f"  SuperBPE 16k                : {1000 / cpt_sup:7,.0f} bước  ({n_byte_nfc / n_char_nfc * cpt_sup:.1f}× ngắn hơn byte)")
print("\nVì chi phí attention tăng theo T (notebook 03 mục 4), chuỗi dài gấp 8 lần không chỉ tốn 8 lần compute"
      "\nở phần matmul mà còn tệ hơn ở phần attention.")

## 10. Tóm tắt

| Thuật toán | Tiêu chí học | Encode | Đặc điểm |
|---|---|---|---|
| BPE | cặp có tần suất cao nhất | áp merge theo thứ tự đã học | đơn giản, xác định, phổ biến nhất |
| SuperBPE | BPE hai giai đoạn, giai đoạn 2 bỏ tách dấu cách | như BPE | token bắc qua dấu cách, ít hơn ~18% token |
| WordPiece | $\mathrm{freq}(ab)/(\mathrm{freq}(a)\mathrm{freq}(b))$ | tham lam dài nhất | thiên về mảnh đặc trưng |
| Unigram | EM + tỉa mảnh đóng góp thấp | Viterbi trên $\sum \log p$ | có phân phối nhiều cách tách, lấy mẫu được |

Hai cái bẫy đã gặp thật trong project:

1. **Thiếu 256 byte trong bảng chữ cái gốc** → byte lạ bị bỏ im lặng → bpc đẹp giả.
2. **Pretokenizer thiếu `\p{M}`** → dấu phụ NFD bị tách khỏi chữ → toàn bộ nhánh NFD vô nghĩa.

## 11. Câu hỏi tự kiểm

1. Vì sao byte-level BPE không bao giờ gặp OOV, trong khi tokenizer mức ký tự vẫn có thể?
2. Pretokenizer quyết định cái gì mà danh sách merge không thể sửa được về sau?
3. SuperBPE cần hai giai đoạn thay vì bỏ tách dấu cách ngay từ đầu vì lý do gì?
4. Với $t = 0{,}9$ và $V = 32\,000$, giai đoạn 1 học bao nhiêu merge, giai đoạn 2 học bao nhiêu?
5. Cho `freq(ab) = 100`, `freq(a) = 1000`, `freq(b) = 120`: BPE và WordPiece xếp cặp này cao hay thấp?
6. Vì sao Unigram lấy mẫu được nhiều cách tách còn BPE thì không?
7. Tăng vocab 16k → 32k làm `lm_head` to ra. Dùng cell mục 1 để nói khi nào việc đó vẫn có lợi.
8. `tokens_per_syllable = 0{,}979` nghĩa là gì, và vì sao nó chưa đủ để kết luận superword là "từ thật"?

**Đáp án gợi ý**

1. Vì mọi văn bản đều là một dãy byte và cả 256 byte đều nằm trong vocab; vocab mức ký tự không thể chứa hết mọi
   code point Unicode.
2. Ranh giới mà không token nào được phép bắc qua — ví dụ regex stage 1 cấm token chứa dấu cách ở giữa.
3. Vì nếu bỏ từ đầu, superword cạnh tranh ngân sách merge với mảnh từ ngay từ bước đầu; lịch hai giai đoạn buộc
   học xong từ vựng cấp từ trước.
4. $0{,}9 \times 32\,000 - 256 = 28\,544$ merge kế thừa, và $3\,200$ merge mới.
5. BPE: cao (tần suất 100). WordPiece: $100/(1000 \times 120) = 8{,}3 \times 10^{-4}$, thấp vì `a` quá phổ biến.
6. Vì Unigram có xác suất cho mọi cách tách, nên lấy mẫu theo phân phối đó; BPE chỉ có một thứ tự merge cố định.
7. Khi mức nén tăng đủ bù phần `lm_head` to thêm — so cột "FLOPs mỗi ký tự", không phải "FLOPs mỗi token".
8. Trung bình một token phủ hơn một âm tiết. Chưa đủ vì token có thể cắt ngang từ ("của một" là hai từ khác
   nhau); phải đo trùng khớp ranh giới từ như H4.

**Nguồn đọc thêm**

- [SuperBPE: Space Travel for Language Models](https://arxiv.org/abs/2503.13423) — mục 2–3.
- [Neural Machine Translation of Rare Words with Subword Units](https://aclanthology.org/P16-1162/) — BPE gốc.
- [Subword Regularization](https://aclanthology.org/P18-1007/) — Unigram và lấy mẫu cách tách.
- [SentencePiece](https://aclanthology.org/D18-2012/) — bản cài đặt và Metaspace.
- [Tokenization Is More Than Compression](https://aclanthology.org/2024.emnlp-main.40/) — vì sao nén tốt hơn
  không tự động nghĩa là model tốt hơn, đúng kết luận H1 của project.
- [How Good is Your Tokenizer?](https://arxiv.org/abs/2012.15613) — mục 3, fertility và ảnh hưởng tới model.
- [Explaining and Mitigating Crosslingual Tokenizer Inequities](https://arxiv.org/abs/2510.21909) — bài MonTok,
  nguồn số CTC tiếng Việt ở mục 8.
- [Karpathy, Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE) — video 2 tiếng, BPE và
  các lỗi thực tế của tokenizer.